In [1]:
## Load packages
import scanpy as sc
import zarr
import re
import numpy as np
import pandas as pd
import os
import gc
from scipy.sparse import issparse
from scipy.sparse import coo_matrix
from scipy.stats import fisher_exact, ranksums
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests
import scipy.sparse as sp
from tqdm import tqdm

/jp_envs/envs/scanpy/lib/python3.12/site-packages/leidenalg/VertexPartition.py:413: SyntaxWarning: invalid escape sequence '\m'
  .. math:: Q = \\frac{1}{m} \\sum_{ij} \\left(A_{ij} - \\frac{k_i^\mathrm{out} k_j^\mathrm{in}}{m} \\right)\\delta(\\sigma_i, \\sigma_j),
/jp_envs/envs/scanpy/lib/python3.12/site-packages/leidenalg/VertexPartition.py:788: SyntaxWarning: invalid escape sequence '\m'
  .. math:: Q = \\sum_{ij} \\left(A_{ij} - \\gamma \\frac{k_i^\mathrm{out} k_j^\mathrm{in}}{m} \\right)\\delta(\\sigma_i, \\sigma_j),
/jp_envs/envs/scanpy/lib/python3.12/site-packages/leidenalg/Optimiser.py:27: SyntaxWarning: invalid escape sequence '\g'
  implementation therefore does not guarantee subpartition :math:`\gamma`-density.
/jp_envs/envs/scanpy/lib/python3.12/site-packages/leidenalg/Optimiser.py:346: SyntaxWarning: invalid escape sequence '\s'
  .. math:: Q = \sum_k \\lambda_k Q_k.


In [2]:
## define DMG calling function
def call_dmgs_wilcox(
    adata,
    target_cells,
    reference_cells,
    min_count=20,
    delta_thresh=0.2,
    fdr=0.05,
    show_progress=True
):
    cell_idx = pd.Index(adata.obs_names)

    target_idx = cell_idx.get_indexer(target_cells)
    target_idx = target_idx[target_idx >= 0]

    ref_idx = cell_idx.get_indexer(reference_cells)
    ref_idx = ref_idx[ref_idx >= 0]

    X = adata.X
    n_features = X.shape[1]

    if sp.issparse(X):
        X = X.toarray()

    pvals = np.ones(n_features)
    stats = np.zeros(n_features)
    mean_target = np.zeros(n_features)
    mean_ref = np.zeros(n_features)

    iterator = tqdm(range(n_features), desc="Running Wilcoxon") if show_progress else range(n_features)

    n_target_total = len(target_idx)
    n_ref_total = len(ref_idx)

    pct_valid_target = np.zeros(n_features)
    pct_valid_ref = np.zeros(n_features)
    for i in iterator:
        t_raw = X[target_idx, i]
        r_raw = X[ref_idx, i]
        t_valid_mask = (t_raw != -1) & (~np.isnan(t_raw))
        r_valid_mask = (r_raw != -1) & (~np.isnan(r_raw))
        pct_valid_target[i] = t_valid_mask.sum() / n_target_total
        pct_valid_ref[i] = r_valid_mask.sum() / n_ref_total
        t_vals = t_raw[t_valid_mask]
        r_vals = r_raw[r_valid_mask]
        if (len(t_vals) + len(r_vals)) < min_count:
            continue
        if (len(t_vals) == 0) or (len(r_vals) == 0):
            continue

        mean_target[i] = np.mean(t_vals)
        mean_ref[i] = np.mean(r_vals)

        stat, p = ranksums(t_vals, r_vals)
        stats[i] = stat
        pvals[i] = p

    _, qvals, _, _ = multipletests(pvals, method="fdr_bh")

    delta = mean_target - mean_ref

    dmgs = (qvals < fdr) & (np.abs(delta) > delta_thresh)

    results = pd.DataFrame({
        "var": adata.var_names,
        "stat": stats,
        "pval": pvals,
        "qval": qvals,
        "mean_target": mean_target,
        "mean_ref": mean_ref,
        "delta": delta,
        "pct_valid_target": pct_valid_target,
        "pct_valid_ref": pct_valid_ref,
        "DMG": dmgs
    })

    return results

## define helper functions
def get_zarr_ncells(zarr_path, tail_size=100000):
    store = zarr.open(zarr_path, mode='r')

    if "row" not in store or store["row"].shape[0] == 0:
        return 0

    row_ds = store["row"]

    if row_ds.shape[0] > tail_size:
        rows = row_ds[-tail_size:]
    else:
        rows = row_ds[:]

    return int(np.max(rows)) + 1

def get_h5ad_cell_progress(sample_ids, h5ad_files):
    records = []
    cum_cells = 0

    for f, sample_id in zip(h5ad_files, sample_ids):
        print(f"Reading: {f}")

        adata = sc.read_h5ad(f, backed='r')
        n_cells = adata.shape[0]

        start_idx = cum_cells
        cum_cells += n_cells
        end_idx = cum_cells

        records.append({
            "file": f,
            "sample_id": sample_id,
            "n_cells": n_cells,
            "cum_cells": cum_cells,
            "start_idx": start_idx,
            "end_idx": end_idx
        })
        print(f"Detecting: {n_cells} cells. Cumulative idx is {end_idx}")
        # cleanup
        adata.file.close()
        del adata
        gc.collect()

    return pd.DataFrame(records)

In [ ]:
## (pre-processing step)
### transpose the per sample methylation impute matrix into chunks of genomic regions (.zarr)
### this step is to convert input methylation matrix to enable chunked DMG calling afterwards

var_dim = "promoter"

chunk_size = 5000  # number of bins per chunk
out_dir = f"tmp_dataset/methy_{var_dim}_impute_chunks_zarr"
h5ad_dir = f"tmp_dataset/h5ad_{var_dim}/" ### change this to the folder containing PRIME-IMPUTE predicted .h5ad methylation matrix
sample_file = "../input_metadata/sample_data.txt"


df_samples = pd.read_csv(sample_file, sep="\t") 
h5ad_files = [
    os.path.join(h5ad_dir, f"{sid}_{var_dim}_imputed.h5ad") 
    for sid in df_samples["sample_id"]
]

processed_df = get_h5ad_cell_progress(df_samples["sample_id"],h5ad_files)

print("Files to process:")
for f in h5ad_files:
    print(f)


os.makedirs(out_dir, exist_ok=True)
adata0 = sc.read_h5ad(h5ad_files[0], backed='r')
n_bins = adata0.shape[1]
var_names = adata0.var_names.values.astype(str)

adata0.file.close()
del adata0
gc.collect()

print(f"Total bins: {n_bins}")

np.savetxt(os.path.join(out_dir, "var_names.txt"), var_names, fmt="%s")

chunk_stores = {}
for start in range(0, n_bins, chunk_size):
    end = min(start + chunk_size, n_bins)
    print(f"Initializing chunk {start}:{end}")
    path = os.path.join(out_dir, f"bins_{start}_{end}.zarr")
    store = zarr.open(path, mode='a')

    if "data" not in store:
        print(f"Initializing new chunk {start}:{end}")
        store.create_dataset("data", shape=(0,), dtype=np.float32, maxshape=(None,), chunks=(100000,))
        store.create_dataset("row", shape=(0,), dtype=np.int32, maxshape=(None,), chunks=(100000,))
        store.create_dataset("col", shape=(0,), dtype=np.int32, maxshape=(None,), chunks=(100000,))
    else:
        print(f"Resuming existing chunk {start}:{end}")

    chunk_stores[(start, end)] = store
all_obs_names = []

for f, sample_id in zip(h5ad_files, df_samples["sample_id"]):
    print(f"\nProcessing file: {f}")
    adata = sc.read_h5ad(f, backed='r')  # open manually
    obs_names = adata.obs_names.values.astype(str)
    all_obs_names.extend(obs_names)

    end_idx = processed_df.loc[processed_df['sample_id'] == sample_id, 'end_idx'].iloc[0]
    
    for (start, end), store in chunk_stores.items():
        print(f"  bins {start}:{end}")
        n_processed = get_zarr_ncells(f"{out_dir}/bins_{start}_{end}.zarr")
        print(f"    n_processed = {n_processed}, end_idx = {end_idx}", flush=True)
        if end_idx <= n_processed:
            print(f"Already processed. Skip chunk.", flush = True)
            continue
        
        sub = adata[:, start:end]
        X = sub.X  # sparse matrix

        n_cells, n_bins = X.shape

        rows, cols = np.indices((n_cells, n_bins))
        rows = rows.ravel()
        cols = cols.ravel()
        data = X.ravel().astype(np.float32, copy=False)

        if store["row"].shape[0] == 0:
            old_n_cells = 0
        else:
            old_n_cells = int(store["row"][:].max() + 1)

        rows = rows + old_n_cells

        for name, arr in zip(["data", "row", "col"], [data, rows, cols]):
            ds = store[name]
            ds.resize(ds.shape[0] + arr.shape[0])
            ds[-arr.shape[0]:] = arr

        print(f"    appended {n_cells} cells × {n_bins} bins")

    adata.file.close()
    del adata
    gc.collect()

np.savetxt(os.path.join(out_dir, "obs_names.txt"), all_obs_names, fmt="%s")

print("\nDone.")

In [4]:
## DMG across cell types
var_dim = "promoter"
cell_meta = pd.read_csv("../input_metadata/cell_meta.tsv", sep="\t")

zarr_dir = f"tmp_dataset/methy_{var_dim}_impute_chunks_zarr/"
var_names = np.loadtxt(os.path.join(zarr_dir, "var_names.txt"),dtype=str)
obs_names = np.loadtxt(os.path.join(zarr_dir, "obs_names.txt"),dtype=str)
bin_files = sorted(os.listdir(zarr_dir))
bin_files = [f for f in bin_files if f.endswith(".zarr")]

cell_types = cell_meta['cell_type'].unique()

output_file = f"tmp_dataset/DMG/DMG_celltype_{var_dim}_imputed_wilcox.tsv"
os.makedirs("tmp_dataset", exist_ok=True)

first_write = True

processed_bin = []

if os.path.exists(output_file):
    first_write = False
    tmp_file = pd.read_csv(output_file,sep="\t")
    max_num = max(tmp_file.value_counts('bin_chunk'))
    bin_counts = tmp_file['bin_chunk'].value_counts()
    #if a bin_chunk has number less than max_num, remove the bin_chunk and save the output_file
    #remove all existing bin_chunk from bin_files
    incomplete_bins = bin_counts[bin_counts < max_num].index.tolist()
    if incomplete_bins:
        print(f"Removing incomplete bin_chunks: {incomplete_bins}")
        tmp_file = tmp_file[~tmp_file['bin_chunk'].isin(incomplete_bins)]
        tmp_file.to_csv(output_file, sep="\t", index=False)
        print("save back")
        # Remove corresponding files from bin_files
        processed_bin = tmp_file['bin_chunk'].unique().tolist()
        print(f"processed bin_files: {len(processed_bin)}")
    else:
        processed_bin = tmp_file['bin_chunk'].unique().tolist()
        print("No deletion in the file is needed.")
        print(f"processed bin_files: {len(processed_bin)}")

for bin_file in bin_files:
    print(f"\nLoading bin chunk: {bin_file}", flush=True)
    bin_file_path = os.path.join(zarr_dir, bin_file)
    if bin_file in processed_bin:
        print(f"{bin_file} already processed. Skip.")
        continue
    
    match = re.search(r"bins_(\d+)_(\d+)\.zarr", bin_file)
    if match is None:
        raise ValueError(f"Filename format unexpected: {bin_file}")

    start = int(match.group(1))
    end = int(match.group(2))

    var_names_chunk = var_names[start:end]

    z = zarr.open(bin_file_path, mode='r')

    X = np.full((len(obs_names), end - start), -1, dtype=np.float32)
    X[z["row"][:], z["col"][:]] = z["data"][:]

    adata_chunk = sc.AnnData(
        X=X,
        obs=pd.DataFrame(index=obs_names),
        var=pd.DataFrame(index=var_names_chunk)
    )

    valid_cells = adata_chunk.obs_names.intersection(cell_meta['cell_id'])
    adata_chunk = adata_chunk[valid_cells].copy()

    for ct in cell_types:
        print(f"Scanning DMGs for cell type: {ct}", flush=True)

        target_cells = cell_meta.loc[cell_meta['cell_type'] == ct, 'cell_id'].values
        reference_cells = cell_meta.loc[cell_meta['cell_type'] != ct, 'cell_id'].values

        dmgs_df = call_dmgs_wilcox(
            adata=adata_chunk,
            target_cells=target_cells,
            reference_cells=reference_cells,
            min_count=20,
            delta_thresh=0.2,
            fdr=0.05,
            show_progress=True
        )

        dmgs_df['bin_chunk'] = bin_file
        dmgs_df['cell_type'] = ct

        dmgs_df.to_csv(
            output_file,
            sep="\t",
            mode="a",
            index=False,
            header=first_write
        )
        first_write = False

        del dmgs_df

    # free memory aggressively (important now!)
    del adata_chunk, X
    import gc
    gc.collect()

print("DMG scanning complete!")

/tmp/ipykernel_10311/786018547.py:3: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  cell_meta = pd.read_csv("tmp_dataset/rna_methy_cell_meta_full.v260417.tsv", sep="\t")



Loading bin chunk: bins_0_2000.zarr
Scanning DMGs for cell type: Epithelial


Running Wilcoxon: 100%|██████████| 2000/2000 [01:41<00:00, 19.75it/s]

Scanning DMGs for cell type: Smooth muscle



Running Wilcoxon: 100%|██████████| 2000/2000 [01:39<00:00, 20.18it/s]

...

Scanning DMGs for cell type: Plasma



Running Wilcoxon: 100%|██████████| 2000/2000 [01:42<00:00, 19.52it/s]

Scanning DMGs for cell type: Schwann & other




unning Wilcoxon: 100%|██████████| 2000/2000 [01:42<00:00, 19.58it/s]

DMG scanning complete!


In [ ]:
## calling DMGs between sublineages

col_name = "mye_cl" ### using Myeloid as an example
var_dim = "promoter"
cell_meta = pd.read_csv("../input_metadata/cell_meta.tsv", sep="\t")
zarr_dir = f"tmp_dataset/methy_{var_dim}_impute_chunks_zarr/"
var_names = np.loadtxt(os.path.join(zarr_dir, "var_names.txt"),dtype=str)
obs_names = np.loadtxt(os.path.join(zarr_dir, "obs_names.txt"),dtype=str)
bin_files = sorted(os.listdir(zarr_dir))
bin_files = [f for f in bin_files if f.endswith(".zarr")]

cell_types = cell_meta['cell_type'].unique()

output_file = "tmp_dataset/DMG/DMG_sub_mye_promoter_imputed_wilcox.tsv"
os.makedirs("tmp_dataset", exist_ok=True)

if os.path.exists(output_file):
    os.remove(output_file)

first_write = True

subset_meta = cell_meta.loc[cell_meta[col_name].notna()]
subclusters = subset_meta[col_name].unique()

for bin_file in bin_files:
    print(f"\nLoading bin chunk: {bin_file}", flush=True)
    bin_file_path = os.path.join(zarr_dir, bin_file)

     match = re.search(r"bins_(\d+)_(\d+)\.zarr", bin_file)
    if match is None:
        raise ValueError(f"Filename format unexpected: {bin_file}")

    start = int(match.group(1))
    end = int(match.group(2))

    var_names_chunk = var_names[start:end]

    z = zarr.open(bin_file_path, mode='r')

    X = np.full((len(obs_names), end - start), -1, dtype=np.float32)
    X[z["row"][:], z["col"][:]] = z["data"][:]

    adata_chunk = sc.AnnData(
        X=X,
        obs=pd.DataFrame(index=obs_names),
        var=pd.DataFrame(index=var_names_chunk)
    )

    valid_cells = adata_chunk.obs_names.intersection(subset_meta['cell_id'])
    adata_chunk = adata_chunk[valid_cells].copy()

    for ct in subclusters:
        print(f"Scanning DMGs for cell type: {ct}", flush=True)

        target_cells = subset_meta.loc[subset_meta[col_name] == ct, 'cell_id'].values
        reference_cells = subset_meta.loc[subset_meta[col_name] != ct, 'cell_id'].values

        dmgs_df = call_dmgs_wilcox(
            adata=adata_chunk,
            target_cells=target_cells,
            reference_cells=reference_cells,
            min_count=20,
            delta_thresh=0.2,
            fdr=0.05,
            show_progress=True
        )

        dmgs_df['bin_chunk'] = bin_file
        dmgs_df['cluster'] = ct

        dmgs_df.to_csv(
            output_file,
            sep="\t",
            mode="a",
            index=False,
            header=first_write
        )
        first_write = False

        del dmgs_df

    del adata_chunk, X
    import gc
    gc.collect()

print("DMG scanning complete!")

In [6]:
## DMG across group

## hard-code G2 vs. G3, G2 vs. G4, G3 vs. G4 for any comparison, including epi BE and C, not LE
var_dim = "promoter"
celltype = "smc"
subcl = "sSMC"

comparisons = [
    (f"G3_{subcl}", f"G2_{subcl}"),
    (f"G4_{subcl}", f"G2_{subcl}"),
    (f"G4_{subcl}", f"G3_{subcl}"),  
]

output_file = f"tmp_dataset/DMG/DMG_sub_{celltype}_{subcl}_group_comparison_{var_dim}_imputed_wilcox.v3.tsv"

cell_meta = pd.read_csv("tmp_dataset/rna_methy_cell_meta_full.v260417.tsv", sep="\t")
cell_meta['cell_group'] = np.nan

cell_meta['cell_group'] = np.where(
    cell_meta[f'{celltype}_cl']==subcl,
    "G" + cell_meta['Group'].astype(str) + f"_{subcl}",
    np.nan
)

subset_meta = cell_meta.dropna(subset=['cell_group'])
#cell_meta['cell_group'].value_counts()

#zarr_dir = "mxy_lishengdi_data/methy_geneslop2k_impute_chunks_zarr/"
zarr_dir = f"mxy_lishengdi_data/methy_{var_dim}_impute_chunks_zarr/"
var_names = np.loadtxt(os.path.join(zarr_dir, "var_names.txt"),dtype=str)
obs_names = np.loadtxt(os.path.join(zarr_dir, "obs_names.txt"),dtype=str)
bin_files = sorted(os.listdir(zarr_dir))
bin_files = [f for f in bin_files if f.endswith(".zarr")]

os.makedirs("tmp_dataset", exist_ok=True)

first_write = True

processed_bin = []

if os.path.exists(output_file):
    first_write = False
    tmp_file = pd.read_csv(output_file,sep="\t")
    max_num = max(tmp_file.value_counts('bin_chunk'))
    bin_counts = tmp_file['bin_chunk'].value_counts()
    #if a bin_chunk has number less than max_num, remove the bin_chunk and save the output_file
    #remove all existing bin_chunk from bin_files
    incomplete_bins = bin_counts[bin_counts < max_num].index.tolist()
    if incomplete_bins:
        print(f"Removing incomplete bin_chunks: {incomplete_bins}")
        tmp_file = tmp_file[~tmp_file['bin_chunk'].isin(incomplete_bins)]
        #tmp_file.to_csv(output_file, sep="\t", index=False)
        print("save back")
        # Remove corresponding files from bin_files
        processed_bin = tmp_file['bin_chunk'].unique().tolist()
        print(f"processed bin_files: {len(processed_bin)}")
    else:
        processed_bin = tmp_file['bin_chunk'].unique().tolist()
        print("No deletion in the file is needed.")
        print(f"processed bin_files: {len(processed_bin)}")


for bin_file in bin_files:
    print(f"\nLoading bin chunk: {bin_file}", flush=True)
    bin_file_path = os.path.join(zarr_dir, bin_file)
    if bin_file in processed_bin:
        print(f"{bin_file} already processed. Skip.")
        continue
    
    match = re.search(r"bins_(\d+)_(\d+)\.zarr", bin_file)
    if match is None:
        raise ValueError(f"Filename format unexpected: {bin_file}")

    start = int(match.group(1))
    end = int(match.group(2))

    var_names_chunk = var_names[start:end]

    z = zarr.open(bin_file_path, mode='r')

    # IMPORTANT: reconstruct dense matrix
    X = np.full((len(obs_names), end - start), -1, dtype=np.float32)
    X[z["row"][:], z["col"][:]] = z["data"][:]

    adata_chunk = sc.AnnData(
        X=X,
        obs=pd.DataFrame(index=obs_names),
        var=pd.DataFrame(index=var_names_chunk)
    )

    # Subset cells
    valid_cells = adata_chunk.obs_names.intersection(subset_meta['cell_id'])
    adata_chunk = adata_chunk[valid_cells].copy()

    for target_group, reference_group in comparisons:
        print(f"Scanning DMGs: {target_group} vs {reference_group}", flush=True)

        target_cells = subset_meta.loc[
            subset_meta["cell_group"] == target_group, 'cell_id'
        ].values

        reference_cells = subset_meta.loc[
            subset_meta["cell_group"] == reference_group, 'cell_id'
        ].values

        # Skip if either group is empty
        if len(target_cells) == 0 or len(reference_cells) == 0:
            print(f"Skip {target_group} vs {reference_group} (empty group)")
            continue

        dmgs_df = call_dmgs_wilcox(
            adata=adata_chunk,
            target_cells=target_cells,
            reference_cells=reference_cells,
            min_count=20,
            delta_thresh=0.2,
            fdr=0.05,
            show_progress=True
        )

        # Add metadata
        dmgs_df['bin_chunk'] = bin_file
        dmgs_df['target'] = target_group
        dmgs_df['reference'] = reference_group

        # Write incrementally
        dmgs_df.to_csv(
            output_file,
            sep="\t",
            mode="a",
            index=False,
            header=first_write
        )
        first_write = False

        del dmgs_df

    # free memory aggressively (important now!)
    del adata_chunk, X
    import gc
    gc.collect()

print("DMG scanning complete!")

/tmp/ipykernel_762/4239608788.py:16: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  cell_meta = pd.read_csv("tmp_dataset/rna_methy_cell_meta_full.v260417.tsv", sep="\t")



Loading bin chunk: bins_0_2000.zarr
Scanning DMGs: G3_sSMC vs G2_sSMC


Running Wilcoxon: 100%|██████████| 2000/2000 [00:01<00:00, 1640.01it/s]

Scanning DMGs: G4_sSMC vs G2_sSMC



Running Wilcoxon: 100%|██████████| 2000/2000 [00:01<00:00, 1651.03it/s]

...

Scanning DMGs: G4_sSMC vs G2_sSMC



Running Wilcoxon: 100%|██████████| 2000/2000 [00:01<00:00, 1607.87it/s]

Scanning DMGs: G4_sSMC vs G3_sSMC




unning Wilcoxon: 100%|██████████| 2000/2000 [00:01<00:00, 1780.33it/s]

DMG scanning complete!


In [ ]:
## calling DMGs between G1, G2, G3 in a given lineage

var_dim = "promoter"
celltype = "epi"
subcl = "LE"

comparisons = [
    ("G2_LE_aneuploid", "G1_LE_euploid"),
    ("G3_LE_aneuploid", "G1_LE_euploid"),
    ("G3_LE_aneuploid", "G2_LE_aneuploid"),
    ("G2_LE_euploid", "G1_LE_euploid"),
    ("G3_LE_euploid", "G1_LE_euploid"),
    ("G3_LE_euploid", "G2_LE_euploid")
]

cell_meta["epi_cl2"] = cell_meta["epi_cl"]

is_le = cell_meta["epi_cl2"] == "LE"

cell_meta.loc[is_le & (cell_meta["ploidy"] == "euploid"), "epi_cl2"] = "LE_euploid"
cell_meta.loc[is_le & (cell_meta["ploidy"] == "aneuploid"), "epi_cl2"] = "LE_aneuploid"

cell_meta = cell_meta[~(is_le & cell_meta["ploidy"].isna())]

output_file = f"tmp_dataset/DMG/DMG_sub_{celltype}_{subcl}_group_comparison_{var_dim}_imputed_wilcox.v3.tsv"

cell_meta = pd.read_csv("tmp_dataset/rna_methy_cell_meta_full.v260417.tsv", sep="\t")
cell_meta['cell_group'] = np.nan

cell_meta['cell_group'] = np.where(
    cell_meta['epi_cl']==subcl,
    "G" + cell_meta['Group'].astype(str) + "_" + cell_meta['epi_cl2'].astype(str),
    np.nan
)

subset_meta = cell_meta.dropna(subset=['cell_group'])

zarr_dir = f"tmp_dataset/methy_{var_dim}_impute_chunks_zarr/"
var_names = np.loadtxt(os.path.join(zarr_dir, "var_names.txt"),dtype=str)
obs_names = np.loadtxt(os.path.join(zarr_dir, "obs_names.txt"),dtype=str)
bin_files = sorted(os.listdir(zarr_dir))
bin_files = [f for f in bin_files if f.endswith(".zarr")]

os.makedirs("tmp_dataset", exist_ok=True)

first_write = True

processed_bin = []

if os.path.exists(output_file):
    first_write = False
    tmp_file = pd.read_csv(output_file,sep="\t")
    max_num = max(tmp_file.value_counts('bin_chunk'))
    bin_counts = tmp_file['bin_chunk'].value_counts()
    incomplete_bins = bin_counts[bin_counts < max_num].index.tolist()
    if incomplete_bins:
        print(f"Removing incomplete bin_chunks: {incomplete_bins}")
        tmp_file = tmp_file[~tmp_file['bin_chunk'].isin(incomplete_bins)]
        print("save back")
        processed_bin = tmp_file['bin_chunk'].unique().tolist()
        print(f"processed bin_files: {len(processed_bin)}")
    else:
        processed_bin = tmp_file['bin_chunk'].unique().tolist()
        print("No deletion in the file is needed.")
        print(f"processed bin_files: {len(processed_bin)}")


for bin_file in bin_files:
    print(f"\nLoading bin chunk: {bin_file}", flush=True)
    bin_file_path = os.path.join(zarr_dir, bin_file)
    if bin_file in processed_bin:
        print(f"{bin_file} already processed. Skip.")
        continue
    
    match = re.search(r"bins_(\d+)_(\d+)\.zarr", bin_file)
    if match is None:
        raise ValueError(f"Filename format unexpected: {bin_file}")

    start = int(match.group(1))
    end = int(match.group(2))

    var_names_chunk = var_names[start:end]

    z = zarr.open(bin_file_path, mode='r')

    X = np.full((len(obs_names), end - start), -1, dtype=np.float32)
    X[z["row"][:], z["col"][:]] = z["data"][:]

    adata_chunk = sc.AnnData(
        X=X,
        obs=pd.DataFrame(index=obs_names),
        var=pd.DataFrame(index=var_names_chunk)
    )

    valid_cells = adata_chunk.obs_names.intersection(subset_meta['cell_id'])
    adata_chunk = adata_chunk[valid_cells].copy()

    for target_group, reference_group in comparisons:
        print(f"Scanning DMGs: {target_group} vs {reference_group}", flush=True)

        target_cells = subset_meta.loc[
            subset_meta["cell_group"] == target_group, 'cell_id'
        ].values

        reference_cells = subset_meta.loc[
            subset_meta["cell_group"] == reference_group, 'cell_id'
        ].values

        if len(target_cells) == 0 or len(reference_cells) == 0:
            print(f"Skip {target_group} vs {reference_group} (empty group)")
            continue

        dmgs_df = call_dmgs_wilcox(
            adata=adata_chunk,
            target_cells=target_cells,
            reference_cells=reference_cells,
            min_count=20,
            delta_thresh=0.2,
            fdr=0.05,
            show_progress=True
        )

        dmgs_df['bin_chunk'] = bin_file
        dmgs_df['target'] = target_group
        dmgs_df['reference'] = reference_group

        dmgs_df.to_csv(
            output_file,
            sep="\t",
            mode="a",
            index=False,
            header=first_write
        )
        first_write = False

        del dmgs_df

    del adata_chunk, X
    import gc
    gc.collect()

print("DMG scanning complete!")